In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import tensorflow as tf
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from keras.utils import to_categorical
from tensorflow.keras.layers import Dense, Dropout, Flatten, Dense, Dropout, Conv2D, MaxPooling2D, Flatten
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor = "val_loss", mode = "min", patience = 10, restore_best_weights=True, verbose=1)

EPOCHS = 1000


In [ ]:
root_path = "/usr/share/MSI25/tumor"

train_dataset = tf.keras.utils.image_dataset_from_directory(
    root_path,
    label_mode="categorical",
    batch_size=32,
    image_size=(100, 100),
    validation_split=0.2,
    subset="training",
    color_mode = "grayscale",
    seed=123,
    shuffle=True
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    root_path,
    label_mode="categorical",
    batch_size=32,
    image_size=(100, 100),
    validation_split=0.2,
    subset="validation",
    color_mode = "grayscale",
    seed=123,
    shuffle=True
)
class_names = train_dataset.class_names   # tutaj, przed .map
print(class_names)

# odpowiednik rescale=1./255 - piksele z zakresu 0-255 na 0-1
train_dataset = train_dataset.map(lambda x, y: (x / 255.0, y))
validation_dataset = validation_dataset.map(lambda x, y: (x / 255.0, y))


In [ ]:
# sprawdzane wartości hiperparametrów
lista_filtrow = [4, 16, 32]
lista_neuronow = [32, 64, 128]
 
wyniki = []
 
for filtry in lista_filtrow:
    for neurony in lista_neuronow:
        print("filtry =", filtry, "neurony =", neurony)
 
        model = Sequential()
        model.add(Conv2D(filters = filtry, kernel_size = (3, 3), activation='relu', input_shape=(100, 100, 1)))
        model.add(MaxPooling2D((2, 2)))
        model.add(Conv2D(filters = filtry, kernel_size = (3, 3), activation='relu'))
        model.add(MaxPooling2D((2, 2)))
        model.add(Flatten())
        model.add(Dense(neurony, activation='relu'))
        model.add(Dense(2, activation='softmax'))
        model.compile(optimizer='adam',
                      loss='categorical_crossentropy',
                      metrics=['accuracy'])
 
        history = model.fit(train_dataset, epochs=EPOCHS, validation_data=(validation_dataset), callbacks = [es], verbose=2)
 
        hist = pd.DataFrame(history.history)
        najlepsza = hist.val_loss.idxmin()          # epoka z najmniejszym val_loss
        wyniki.append({"filtry": filtry,
                       "neurony": neurony,
                       "epoki": len(hist),
                       "val_loss": hist.val_loss[najlepsza],
                       "val_accuracy": hist.val_accuracy[najlepsza]})
 
wyniki = pd.DataFrame(wyniki).sort_values("val_loss")
print(wyniki)
wyniki.to_csv("wyniki_strojenia.csv", index=False)